# Scope_Check — is every domain with content classified?

Release check for the consumer tracks. Run it after a CT or COSMoS refresh, before the
four Findings consumers.

The consumers take their scope from `SDTM_Domain_Metadata.xlsx`
(`Immunogenicity`, `Specimen_Based`, `Measurement`, `Instrument_Domain_Rules`) minus named exclusions
in each notebook. A domain that is missing from the metadata can therefore never
enter a consumer, and nothing would say so. This notebook makes that visible.

Every domain that carries content is listed:

- SDTM CT test codes (`SDTM_Test_Identity.xlsx`, `SDTM_Instrument_Test_Identity.xlsx`)
- COSMoS Dataset Specializations (`COSMoS_Graph.xlsx`, `DSS` sheet)

and checked against two lists:

- `SDTM_Domain_Metadata.xlsx` — the repo's own domain reference (public sources)
- the SDTM Domain Abbreviation codelist C66734 in the SDTM CT file — CDISC's published
  list of domain codes

**Fails** when a domain with content is missing from either list. A domain that is
classified but has no consumer class (e.g. SC, TU) is reported, not failed: that is a
decision recorded in the metadata.

Section 5 checks the metadata itself against SDTMIG v3.4 (variable table and table of
contents, gitignored local copies in `sdtm-domain-reference/downloads/`): every IG dataset
is listed once, in `Domains` or in `Non_Observation_Datasets`, and `SDTMIG_Section`, `IG_Name`,
`IG_Class`, `Has_Test_Codes` and `Specimen_Based` agree with the IG. **Fails** on any difference.

## 1. Setup

In [ ]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent          # sdtm-findings-graph/
REPO_ROOT = BASE_DIR.parent           # cdisc-for-ai/

DOMAIN_META_FILE = REPO_ROOT / 'sdtm-domain-reference' / 'machine_actionable' / 'SDTM_Domain_Metadata.xlsx'
TEST_IDENTITY_FILE = REPO_ROOT / 'sdtm-test-codes' / 'machine_actionable' / 'SDTM_Test_Identity.xlsx'
INSTR_TEST_FILE = REPO_ROOT / 'sdtm-test-codes' / 'machine_actionable' / 'SDTM_Instrument_Test_Identity.xlsx'
SDTM_CT_FILE = REPO_ROOT / 'sdtm-test-codes' / 'downloads' / 'SDTM_Terminology.txt'
GRAPH_FILE = REPO_ROOT / 'cosmos-graph' / 'interim' / 'COSMoS_Graph.xlsx'

DOMAIN_CODELIST = 'C66734'            # SDTM Domain Abbreviation

for f in [DOMAIN_META_FILE, TEST_IDENTITY_FILE, INSTR_TEST_FILE, SDTM_CT_FILE, GRAPH_FILE]:
    if not f.exists():
        raise FileNotFoundError(f)
    print(f'  {f.relative_to(REPO_ROOT)}')

## 2. Load

In [ ]:
meta = pd.read_excel(DOMAIN_META_FILE, sheet_name='Domains', dtype=str).fillna('')
instr_rules = pd.read_excel(DOMAIN_META_FILE, sheet_name='Instrument_Domain_Rules', dtype=str)
test_identity = pd.read_excel(TEST_IDENTITY_FILE, sheet_name='Test Codes', dtype=str).fillna('')
instr_tests = pd.read_excel(INSTR_TEST_FILE, sheet_name='Test Codes', dtype=str).fillna('')
dss = pd.read_excel(GRAPH_FILE, sheet_name='DSS', dtype=str).fillna('')
sdtm_ct = pd.read_csv(SDTM_CT_FILE, sep='\t', dtype=str, keep_default_na=False)

ct_domains = sdtm_ct[sdtm_ct['Codelist Code'] == DOMAIN_CODELIST][['CDISC Submission Value', 'CDISC Synonym(s)']]
ct_domains = ct_domains.rename(columns={'CDISC Submission Value': 'Domain', 'CDISC Synonym(s)': 'CT_Domain_Name'})
if ct_domains.empty:
    raise ValueError(f'Codelist {DOMAIN_CODELIST} not found in {SDTM_CT_FILE.name}')

print(f'SDTM_Domain_Metadata domains:          {len(meta)}')
print(f'C66734 SDTM Domain Abbreviation terms: {len(ct_domains)}')

## 3. Domains with content

In [ ]:
rows = []
for s in test_identity['SDTM_Domains']:
    for d in s.split(';'):
        if d.strip():
            rows.append(('CT_Test_Codes', d.strip()))
rows += [('CT_Instrument_Test_Codes', d) for d in instr_tests['Domain'] if d]
rows += [('COSMoS_DSS', d) for d in dss['domain'] if d]

content = pd.DataFrame(rows, columns=['source', 'Domain'])
content = content.groupby(['Domain', 'source']).size().unstack(fill_value=0).reset_index()
for col in ['CT_Test_Codes', 'CT_Instrument_Test_Codes', 'COSMoS_DSS']:
    if col not in content.columns:
        content[col] = 0

instrument_domains = set(instr_rules['Domain'])

def consumer_class(row):
    if row['Immunogenicity'] == 'Yes':
        return 'Immunogenicity'
    if row['Specimen_Based'] == 'Yes':
        return 'Specimen'
    if row['Measurement'] == 'Yes':
        return 'Measurement'
    if row['Domain'] in instrument_domains:
        return 'Instrument'
    return ''

meta['Consumer_Class'] = meta.apply(consumer_class, axis=1)

report = content.merge(meta[['Domain', 'Domain_Name', 'Observation_Class', 'Consumer_Class']],
                       on='Domain', how='left', indicator='in_meta')
report['In_Domain_Metadata'] = (report['in_meta'] == 'both').map({True: 'Yes', False: 'No'})
report = report.drop(columns='in_meta').merge(ct_domains, on='Domain', how='left')
report['In_CT_C66734'] = report['CT_Domain_Name'].notna().map({True: 'Yes', False: 'No'})
report = report.fillna('')
report = report[['Domain', 'Domain_Name', 'CT_Domain_Name', 'Observation_Class', 'Consumer_Class',
                 'CT_Test_Codes', 'CT_Instrument_Test_Codes', 'COSMoS_DSS',
                 'In_Domain_Metadata', 'In_CT_C66734']].sort_values('Domain').reset_index(drop=True)

with pd.option_context('display.max_rows', None, 'display.width', 200):
    print(report.to_string(index=False))

## 4. Result

In [ ]:
no_class = report[(report['In_Domain_Metadata'] == 'Yes') & (report['Consumer_Class'] == '')]
print(f'Classified, no consumer class (reported, not failed): {", ".join(no_class["Domain"])}')

not_in_meta = report[report['In_Domain_Metadata'] == 'No']
not_in_ct = report[report['In_CT_C66734'] == 'No']

problems = []
if len(not_in_meta):
    problems.append(f'Not in SDTM_Domain_Metadata ({len(not_in_meta)}): {", ".join(not_in_meta["Domain"])}')
if len(not_in_ct):
    problems.append(f'Not a domain code in CT C66734 ({len(not_in_ct)}): {", ".join(not_in_ct["Domain"])}')

if problems:
    raise ValueError('Scope check FAILED\n  ' + '\n  '.join(problems))
print('Scope check PASSED — every domain with content is classified and is a published domain code.')

## 5. Against SDTMIG v3.4

The table of contents gives section and name; the variable table gives class and whether the
domain has a `--TESTCD` variable. `Specimen_Based` = Yes exactly for the domains under 6.3.5
Specimen-based Findings Domains. MB, MS, PC, PP and SUPPQUAL have no table-of-contents entry of
their own (group headings 6.3.5.7, 6.3.5.9; 8.4.1 SUPP--), so their section is stated in the cell
and their name is not checked.

In [ ]:
import re
from pypdf import PdfReader

SDTMIG_CSV = REPO_ROOT / 'sdtm-domain-reference' / 'downloads' / 'SDTMIG_v3.4.csv'
SDTMIG_PDF = REPO_ROOT / 'sdtm-domain-reference' / 'downloads' / 'SDTMIG v3.4-FINAL_2022-07-21.pdf'
for f in [SDTMIG_CSV, SDTMIG_PDF]:
    if not f.exists():
        raise FileNotFoundError(f'{f} (gitignored; download with a cdisc.org sign-in)')

# Datasets the table of contents lists only under a group heading, with no "(XX)" entry of their own
GROUP_SECTIONS = {'MB': '6.3.5.7', 'MS': '6.3.5.7', 'PC': '6.3.5.9', 'PP': '6.3.5.9', 'SUPPQUAL': '8.4.1'}

non_obs = pd.read_excel(DOMAIN_META_FILE, sheet_name='Non_Observation_Datasets', dtype=str).fillna('')
ig = pd.read_csv(SDTMIG_CSV, dtype=str).fillna('')
ig_class = ig.groupby('Dataset Name')['Class'].first()
ig_testcd = set(ig.loc[ig['Variable Name'] == ig['Dataset Name'] + 'TESTCD', 'Dataset Name'])

toc = {}
for page in PdfReader(SDTMIG_PDF).pages[1:8]:
    for line in (page.extract_text() or '').splitlines():
        m = re.match(r'^\s*(\d+(?:\.\d+)+)\s+(.+?)\s*\(([A-Z]{2,8})\)\s*\.{2,}', line)
        if m and m.group(3) not in toc:
            toc[m.group(3)] = (m.group(1), m.group(2).strip())

print(f'SDTMIG v3.4 datasets: {len(ig_class)}   TOC entries with a domain code: {len(toc)}')
print(f'Domains: {len(meta)}   Non_Observation_Datasets: {len(non_obs)}')

ig_problems = []
listed = list(meta['Domain']) + list(non_obs['Dataset'])
twice = sorted({d for d in listed if listed.count(d) > 1})
if twice:
    ig_problems.append(f'In both Domains and Non_Observation_Datasets: {", ".join(twice)}')
missing = sorted(set(ig_class.index) - set(listed))
if missing:
    ig_problems.append(f'In SDTMIG v3.4, in neither Domains nor Non_Observation_Datasets: {", ".join(missing)}')
not_in_ig = sorted(set(meta.loc[meta['SDTMIG_Section'] != '', 'Domain']).union(non_obs['Dataset']) - set(ig_class.index))
if not_in_ig:
    ig_problems.append(f'Listed with an SDTMIG_Section but not in SDTMIG v3.4: {", ".join(not_in_ig)}')


def check_ig(d, section_listed, name_listed, class_listed):
    if d in GROUP_SECTIONS:
        section, name = GROUP_SECTIONS[d], None
    elif d in toc:
        section, name = toc[d]
    else:
        ig_problems.append(f'{d}: not in the SDTMIG v3.4 table of contents and not in GROUP_SECTIONS')
        return None
    if section_listed != section:
        ig_problems.append(f'{d}: SDTMIG_Section {section_listed or "(empty)"}, table of contents {section}')
    if name is not None and name_listed.lower() != name.lower():
        ig_problems.append(f'{d}: IG_Name {name_listed!r}, table of contents {name!r}')
    if class_listed != ig_class[d]:
        ig_problems.append(f'{d}: IG_Class {class_listed or "(empty)"}, SDTMIG v3.4 {ig_class[d]}')
    return section


for _, row in meta[meta['Domain'].isin(ig_class.index)].iterrows():
    d = row['Domain']
    section = check_ig(d, row['SDTMIG_Section'], row['IG_Name'], row['IG_Class'])
    if section is None:
        continue
    if (row['Has_Test_Codes'] == 'Yes') != (d in ig_testcd):
        ig_problems.append(f'{d}: Has_Test_Codes {row["Has_Test_Codes"]}, SDTMIG v3.4 {d}TESTCD {"present" if d in ig_testcd else "absent"}')
    if (row['Specimen_Based'] == 'Yes') != section.startswith('6.3.5.'):
        ig_problems.append(f'{d}: Specimen_Based {row["Specimen_Based"]}, section {section}')

for _, row in non_obs[non_obs['Dataset'].isin(ig_class.index)].iterrows():
    check_ig(row['Dataset'], row['SDTMIG_Section'], row['IG_Name'], row['IG_Class'])

if ig_problems:
    raise ValueError(f'SDTMIG v3.4 check FAILED ({len(ig_problems)})\n  ' + '\n  '.join(ig_problems))
print('SDTMIG v3.4 check PASSED — every IG dataset is listed once; sections, IG names and classes, '
      'Has_Test_Codes and Specimen_Based agree with SDTMIG v3.4.')